# Faz 4 — nnDetection 3B Eğitimi (Fold 0)

**Amaç:** Faz 3'teki 2B YOLO baseline'a karşılık 3B volumetric nesne saptama uygulamak. nnDetection (Baumgartner 2021), nnU-Net protokolünü medikal nesne saptamaya taşıyan self-configuring çerçevedir.

**Adımlar**
1. Ortam + nnDetection kontrolü
2. DICOM seri → NIfTI 3B hacim dönüşümü
3. 2B BBox → 3B kutu yükseltme (ardışık-kesit gruplamayla)
4. Task100_Abdomen veri formatı + dataset.json
5. nnDetection plan + preprocess
6. Fold 0 eğitim
7. Val 3B çıkarım + 2D-projeksiyon F1@IoU

**Kaynak uyarısı:** Kaggle T4 (15GB) için patch_size=96³ önerilir; A100 (40GB) ile 128³ veya 160³ mümkün.

## 1. Ortam ve nnDetection Kontrolü

In [2]:
import os, sys, shutil, subprocess
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

DATA_ROOT = Path(os.environ.get("TR_ABDOMEN_BASE", r"D:/makale-pdf/Proje/abdomen"))
PROJE     = Path(os.environ.get("TR_ABDOMEN_PROJE", r"D:/makale-pdf/Proje"))
CODE      = Path(os.environ.get("TR_ABDOMEN_CODE",  r"D:/makale-pdf/Proje/code"))
EGITIM_DIR = Path(os.environ.get("ABDOMEN_TRAIN_DIR", DATA_ROOT / "Egitim Verisi"))
YARISMA_DIR = Path(os.environ.get("ABDOMEN_TEST_DIR", DATA_ROOT / "Test Verisi"))

os.environ["ABDOMEN_PROJECT_ROOT"] = str(PROJE)
os.environ["ABDOMEN_DATA_ROOT"]    = str(DATA_ROOT)
os.environ["ABDOMEN_TRAIN_DIR"]    = str(EGITIM_DIR)
os.environ["ABDOMEN_TEST_DIR"]     = str(DATA_ROOT / "Yarışma Veri Seti")
os.environ["ABDOMEN_BILGI_XLSX"]   = str(DATA_ROOT / "Bilgi.xlsx")
os.environ["ABDOMEN_OUT_DIR"]      = str(PROJE / "outputs")

sys.path.insert(0, str(CODE))
from src.config import OUT_DIR, SPLIT_DIR, SUPER_CLASSES

NND_ROOT = OUT_DIR / "nndet"
os.environ["det_data"]    = str(NND_ROOT / "raw")
os.environ["det_models"]  = str(NND_ROOT / "models")
os.environ["OMP_NUM_THREADS"] = "1"
for p in [NND_ROOT, Path(os.environ["det_data"]), Path(os.environ["det_models"])]:
    p.mkdir(parents=True, exist_ok=True)
print("det_data  :", os.environ["det_data"])
print("det_models:", os.environ["det_models"])

det_data  : D:\makale-pdf\Proje\outputs\nndet\raw
det_models: D:\makale-pdf\Proje\outputs\nndet\models


In [3]:
try:
    import nndet
    print("nnDetection yüklü:", getattr(nndet, "__version__", "ok"))
except ImportError:
    print("⚠️  nndet yüklü değil. Kurulum:")
    print("   pip install nndet")
    print("   # veya: git clone https://github.com/MIC-DKFZ/nnDetection.git && pip install -e nnDetection")
    print("Windows: Docker veya WSL2 önerilir.")

⚠️  nndet yüklü değil. Kurulum:
   pip install nndet
   # veya: git clone https://github.com/MIC-DKFZ/nnDetection.git && pip install -e nnDetection
Windows: Docker veya WSL2 önerilir.


## 2. DICOM Seri → NIfTI Hacim

In [ ]:
# !pip install SimpleITK --only-binary=:all:

In [4]:
import SimpleITK as sitk


In [5]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from src.splits import load_fold
import pandas as pd

NIFTI_DIR = NND_ROOT / "nifti"
NIFTI_DIR.mkdir(parents=True, exist_ok=True)
fold = 0
fold_cases = sorted(set(load_fold(fold, "train") + load_fold(fold, "val")))
print(f"Fold {fold}: {len(fold_cases)} vaka dönüştürülecek")

def dicom_to_nifti(case_dir, out_path):
    if out_path.exists(): return "skip"
    reader = sitk.ImageSeriesReader()
    names = reader.GetGDCMSeriesFileNames(str(case_dir))
    if not names: return "no_dcm"
    reader.SetFileNames(names)
    try:
        img = reader.Execute()
        sitk.WriteImage(img, str(out_path))
        return "ok"
    except Exception as e:
        return f"err:{e}"

def _conv(case):
    cd = EGITIM_DIR / str(case)
    op = NIFTI_DIR / f"case_{case:05d}_0000.nii.gz"
    return case, dicom_to_nifti(cd, op)

with ThreadPoolExecutor(max_workers=4) as ex:
    results = list(tqdm(ex.map(_conv, fold_cases), total=len(fold_cases), desc="DICOM→NIfTI"))

ok = sum(1 for _, s in results if s == "ok")
skip = sum(1 for _, s in results if s == "skip")
err = [(c, s) for c, s in results if s not in ("ok", "skip")]
print(f"\n✓ Yeni: {ok}, ⏭️  Atlandı: {skip}, ❌ Hata: {len(err)}")
if err: print("Hatalar:", err[:5])

Fold 0: 554 vaka dönüştürülecek


DICOM→NIfTI: 100%|██████████| 554/554 [36:20<00:00,  3.94s/it] 


✓ Yeni: 553, ⏭️  Atlandı: 0, ❌ Hata: 1
Hatalar: [(20317, 'err:Exception thrown in SimpleITK ImageSeriesReader_Execute: D:\\a\\SimpleITK\\SimpleITK\\bld\\ITK-prefix\\include\\ITK-5.4\\itkImageFileReader.hxx:338:\nImageIO returns IO region that does not fully contain the requested region. Requested region: ImageRegion (000000952F07B490)\n  Dimension: 3\n  Index: [0, 0, 0]\n  Size: [608, 512, 1]\nStreamableRegion region: ImageRegion (000000952F07B500)\n  Dimension: 3\n  Index: [0, 0, 0]\n  Size: [512, 512, 1]\n')]


## 3. 2B BBox → 3B Kutu Yükseltme

`Bilgi.xlsx`'teki tüm annotasyonlar 2B. 3B detection için ardışık kesit BBox'larını birleştirip 3B kutu üretiyoruz:
- Aynı vaka + aynı sınıf
- Ardışık veya yakın kesit (Δz ≤ 2)
- 2B IoU ≥ 0.3 → aynı lezyon

In [7]:
import numpy as np
import pydicom

def _2d_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw * ih
    ua = (ax2-ax1)*(ay2-ay1); ub = (bx2-bx1)*(by2-by1)
    return inter / max(ua + ub - inter, 1e-6)

def case_image_id_to_z(case):
    cd = EGITIM_DIR / str(case)
    files = sorted(
        [p for p in cd.glob("*.dcm") if not p.stem.startswith(".")],
        key=lambda p: int(p.stem)
    )
    positions = []
    for p in files:
        ds = pydicom.dcmread(str(p), stop_before_pixels=True)
        zpos = float(getattr(ds, 'ImagePositionPatient', [0,0,int(p.stem)])[2])
        positions.append((int(p.stem), zpos))
    positions.sort(key=lambda x: x[1])
    return {img_id: idx for idx, (img_id, _) in enumerate(positions)}

def lift_2d_to_3d(manifest, case, delta_z=2, iou_th=0.3):
    z_map = case_image_id_to_z(case)
    sub = manifest[(manifest['case'] == case) &
                   (manifest['bboxes'].fillna('').astype(str) != '')]
    items = []
    for _, row in sub.iterrows():
        z = z_map.get(int(row['image_id']))
        if z is None: continue
        for bb_str in str(row['bboxes']).split('|'):
            parts = bb_str.split(',')
            if len(parts) < 5: continue
            sid = int(parts[0]); x1,y1,x2,y2 = map(int, parts[1:5])
            items.append((sid, x1, y1, x2, y2, z))
    boxes_3d = []
    for sid in set(it[0] for it in items):
        cls_items = sorted([it for it in items if it[0] == sid], key=lambda x: x[5])
        used = set()
        for i, it in enumerate(cls_items):
            if i in used: continue
            grp = [it]; used.add(i)
            for j in range(i+1, len(cls_items)):
                if j in used: continue
                last = grp[-1]
                if cls_items[j][5] - last[5] <= delta_z:
                    if _2d_iou(last[1:5], cls_items[j][1:5]) >= iou_th:
                        grp.append(cls_items[j]); used.add(j)
                else: break
            xs1 = min(g[1] for g in grp); ys1 = min(g[2] for g in grp)
            xs2 = max(g[3] for g in grp); ys2 = max(g[4] for g in grp)
            zs1 = min(g[5] for g in grp); zs2 = max(g[5] for g in grp)
            boxes_3d.append({"class": sid, "x1": xs1, "y1": ys1, "z1": zs1,
                             "x2": xs2, "y2": ys2, "z2": zs2,
                             "n_slices": len(grp)})
    return boxes_3d

manifest = pd.read_csv(SPLIT_DIR / "manifest.csv")
demo_case = fold_cases[0]
demo_boxes = lift_2d_to_3d(manifest, demo_case)
print(f"Vaka {demo_case}: {len(demo_boxes)} adet 3B kutu")
for b in demo_boxes[:5]:
    print(f"  sınıf {b['class']} ({SUPER_CLASSES[b['class']]}): "
          f"({b['x1']},{b['y1']},{b['z1']}) → ({b['x2']},{b['y2']},{b['z2']}), {b['n_slices']} kesit")

Vaka 20001: 1 adet 3B kutu
  sınıf 1 (kidney_ureter_stone): (251,290,11) → (262,302,12), 2 kesit


## 4. nnDetection Veri Formatı (Task100_Abdomen)

In [8]:
import json
TASK_NAME = "Task100_Abdomen"
TASK_DIR = Path(os.environ["det_data"]) / TASK_NAME / "raw_splitted"
(TASK_DIR / "imagesTr").mkdir(parents=True, exist_ok=True)
(TASK_DIR / "imagesTs").mkdir(parents=True, exist_ok=True)
(TASK_DIR / "labelsTr").mkdir(parents=True, exist_ok=True)

train_cases = load_fold(fold, "train")
val_cases   = load_fold(fold, "val")

def link_or_copy(src, dst):
    if dst.exists(): return
    try: os.symlink(src, dst)
    except (OSError, NotImplementedError): shutil.copy2(src, dst)

def prep_case(case, split):
    src_nii = NIFTI_DIR / f"case_{case:05d}_0000.nii.gz"
    if not src_nii.exists(): return f"missing:{case}"
    target = TASK_DIR / ("imagesTr" if split != "test" else "imagesTs") / f"case_{case:05d}_0000.nii.gz"
    link_or_copy(src_nii, target)
    if split == "test": return "ok"
    boxes_3d = lift_2d_to_3d(manifest, case)
    label_json = {
        "instances": {str(i+1): {"class": b["class"]} for i, b in enumerate(boxes_3d)},
        "boxes": [[b["x1"], b["y1"], b["z1"], b["x2"], b["y2"], b["z2"]] for b in boxes_3d],
        "classes": [b["class"] for b in boxes_3d],
    }
    (TASK_DIR / "labelsTr" / f"case_{case:05d}.json").write_text(json.dumps(label_json))
    return "ok"

for c in tqdm(train_cases + val_cases, desc="prep train"):
    prep_case(c, "train")
print(f"✓ {len(train_cases) + len(val_cases)} eğitim vakası hazır")

prep train: 100%|██████████| 554/554 [06:41<00:00,  1.38it/s]

✓ 554 eğitim vakası hazır


In [9]:
dataset_meta = {
    "task": TASK_NAME,
    "name": "TR_ABDOMEN_RAD_EMERGENCY",
    "target_class": None,
    "test_labels": False,
    "labels": {str(i): name for i, name in enumerate(SUPER_CLASSES)},
    "modalities": {"0": "CT"},
    "dim": 3,
}
(TASK_DIR.parent / "dataset.json").write_text(json.dumps(dataset_meta, indent=2))
print("dataset.json yazıldı:", TASK_DIR.parent / "dataset.json")
print(json.dumps(dataset_meta, indent=2, ensure_ascii=False))

dataset.json yazıldı: D:\makale-pdf\Proje\outputs\nndet\raw\Task100_Abdomen\dataset.json
{
  "task": "Task100_Abdomen",
  "name": "TR_ABDOMEN_RAD_EMERGENCY",
  "target_class": null,
  "test_labels": false,
  "labels": {
    "0": "acute_cholecystitis",
    "1": "kidney_ureter_stone",
    "2": "acute_pancreatitis",
    "3": "aortic_aneurysm_dissection",
    "4": "acute_appendicitis",
    "5": "acute_diverticulitis"
  },
  "modalities": {
    "0": "CT"
  },
  "dim": 3
}


## 5. nnDetection Plan + Preprocess

```bash
nndet_prep 100 --full_check
```

Patch boyutu, batch boyutu, foreground/background oranını otomatik belirler ve `splits_final.pkl` üretir.

In [ ]:
print("nnDetection preprocess başlatılıyor...")
try:
    r = subprocess.run(["nndet_prep", "100", "--full_check"],
                       capture_output=True, text=True, timeout=3600*3, env=os.environ)
    print("STDOUT:\n", r.stdout[-2000:])
    if r.returncode != 0:
        print("STDERR:\n", r.stderr[-1500:])
except FileNotFoundError:
    print("⚠️  nndet_prep komutu bulunamadı.")
    print("Manuel: shell'de 'nndet_prep 100 --full_check' çalıştırın.")
except subprocess.TimeoutExpired:
    print("Süre aşıldı; arka planda devam ediyor olabilir.")

## 6. Fold 0 Eğitim

In [ ]:
print("nnDetection train — fold 0, ~8-15 saat (T4)...")
try:
    r = subprocess.run(["nndet_train", "100", "--sweep", "-o", "exp.fold=0"],
                       capture_output=False, text=True, env=os.environ)
    print("Eğitim çıktı kodu:", r.returncode)
except FileNotFoundError:
    print("⚠️  Manuel: nndet_train 100 --sweep -o exp.fold=0")

## 7. 3B Çıkarım

In [ ]:
print("nnDetection predict — fold 0...")
try:
    r = subprocess.run(["nndet_predict", "100", "RetinaUNetV001_D3V001_3d",
                        "--fold", "0"],
                       capture_output=False, text=True, env=os.environ)
    print("Predict çıktı kodu:", r.returncode)
except FileNotFoundError:
    print("⚠️  Manuel çalıştırın.")

In [ ]:
from src.evaluation import top5_f1_mean, f1_at_iou
from PIL import Image
from src.config import DET_DATA_DIR

PRED_DIR = Path(os.environ["det_models"]) / TASK_NAME / "RetinaUNetV001_D3V001_3d" / "fold0" / "val_predictions"
print("Tahmin klasörü:", PRED_DIR, "→ var?", PRED_DIR.exists() if PRED_DIR.parent.exists() else "henüz yok")

def load_3d_preds_as_2d(pred_dir):
    rows = []
    if not pred_dir.exists(): return pd.DataFrame(rows)
    for pkl in pred_dir.glob("*_boxes.json"):
        case = int(pkl.stem.split("_")[1])
        data = json.loads(pkl.read_text())
        for b, s, c in zip(data.get("boxes",[]), data.get("scores",[]), data.get("classes",[])):
            x1,y1,z1,x2,y2,z2 = b
            for z in range(int(z1), int(z2)+1):
                rows.append({"case": case, "image_id": z, "class": int(c),
                             "score": float(s),
                             "x1": x1, "y1": y1, "x2": x2, "y2": y2})
    return pd.DataFrame(rows)

pred_df = load_3d_preds_as_2d(PRED_DIR)
print(f"3B→2D projeksiyon: {len(pred_df):,} satır")

In [ ]:
val_lbl_dir = DET_DATA_DIR / f"fold{fold}" / "labels" / "val"
val_img_dir = DET_DATA_DIR / f"fold{fold}" / "images" / "val"
gt_rows = []
for lp in val_lbl_dir.glob("*.txt"):
    ip = val_img_dir / (lp.stem + ".png")
    if not ip.exists(): continue
    W, H = Image.open(ip).size
    stem = lp.stem
    case, image_id = (stem.split("_", 1) + ["0"])[:2]
    for line in lp.read_text().strip().splitlines():
        p = line.split()
        if len(p) < 5: continue
        cl = int(p[0]); cx,cy,w,h = map(float, p[1:5])
        gt_rows.append({"case": int(case), "image_id": int(image_id),
                        "class": cl,
                        "x1": (cx-w/2)*W, "y1": (cy-h/2)*H,
                        "x2": (cx+w/2)*W, "y2": (cy+h/2)*H})
gt_df = pd.DataFrame(gt_rows)
print(f"GT: {len(gt_df):,}")

if not pred_df.empty:
    result = top5_f1_mean(pred_df, gt_df)
    detail = f1_at_iou(pred_df, gt_df, iou_th=0.3)
    print(f"\nnnDetection fold 0 — Top-5 F1: {result['top5_mean_f1']:.4f}")
    print(f"Macro F1 @ IoU 0.3: {detail['macro_f1']:.4f}")
else:
    print("Tahmin yok — predict adımının bitmesini bekleyin.")

## 8. Faz 4 Çıktı Özeti

| Çıktı | Yol |
|---|---|
| NIfTI hacimler | `outputs/nndet/nifti/case_*.nii.gz` |
| Task verisi | `outputs/nndet/raw/Task100_Abdomen/raw_splitted/` |
| Eğitilmiş model | `outputs/nndet/models/Task100_Abdomen/RetinaUNetV001_D3V001_3d/fold0/` |
| 3B tahminler | `.../fold0/val_predictions/*_boxes.json` |

**Sonraki:** `Faz5_Harici_Test_Yarisma.ipynb` (YOLO 2B vs nnDetection 3B karşılaştırma).